In [1]:
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon, t as t_dist
from statsmodels.stats.multitest import multipletests
from itertools import combinations

In [2]:
kfold_results_df = pd.read_pickle('all_kfold_results.pkl')

In [3]:
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon, t as t_dist
from statsmodels.stats.multitest import multipletests
from itertools import combinations

# ============================================================
# Statistical Analysis Functions
# ============================================================

def friedman_test(recall_matrix):
    model_arrays = [recall_matrix[col].values for col in recall_matrix.columns]
    stat, p_value = friedmanchisquare(*model_arrays)
    n, k = recall_matrix.shape
    kendalls_w = stat / (n * (k - 1))
    return {'chi2': stat, 'p': p_value, 'w': kendalls_w, 'n': n, 'k': k}

def pairwise_wilcoxon_holm(recall_matrix):
    models = recall_matrix.columns.tolist()
    pairs = list(combinations(models, 2))
    p_vals = []
    effect_sizes = []

    for m1, m2 in pairs:
        try:
            # Wilcoxon signed-rank test
            res = wilcoxon(recall_matrix[m1], recall_matrix[m2])
            p = res.pvalue

            # Rank-biserial correlation: (W+ - W-) / (W+ + W-)
            # scipy.stats.wilcoxon returns the smaller sum of ranks as statistic by default
            # For n=5, total sum of ranks is 1+2+3+4+5 = 15
            n = len(recall_matrix)
            total_ranks = n * (n + 1) / 2
            # We calculate effect size manually based on signed ranks
            diff = recall_matrix[m1] - recall_matrix[m2]
            ranks = pd.Series(diff).abs().rank()
            w_plus = (ranks[diff > 0]).sum()
            w_minus = (ranks[diff < 0]).sum()
            r = (w_plus - w_minus) / total_ranks
        except:
            p = 1.0
            r = 0.0
        p_vals.append(p)
        effect_sizes.append(r)

    # Apply Holm-Bonferroni correction
    reject, p_adj, _, _ = multipletests(p_vals, method='holm')

    return pd.DataFrame({
        'Model 1': [p[0] for p in pairs],
        'Model 2': [p[1] for p in pairs],
        'Rank-biserial r': effect_sizes,
        'p_raw': p_vals,
        'p_adj': p_adj,
        'Significant': reject
    })

def get_summary_with_ci(df, confidence=0.95):
    n = 5
    t_crit = t_dist.ppf((1 + confidence) / 2, df=n-1)
    summary = df.groupby('Model')['Recall'].agg(['mean', 'std']).reset_index()
    summary['CI_margin'] = t_crit * (summary['std'] / np.sqrt(n))
    summary['CI_Lower'] = summary['mean'] - summary['CI_margin']
    summary['CI_Upper'] = summary['mean'] + summary['CI_margin']
    return summary.sort_values('mean', ascending=False)

# ============================================================
# Execution
# ============================================================

analysis_df = kfold_results_df.copy()
recall_matrix = analysis_df.pivot(index='Fold', columns='Model', values='Recall')

print("=== Mean Recall with 95% Confidence Intervals ===")
display(get_summary_with_ci(analysis_df))

res = friedman_test(recall_matrix)
print(f"\n=== Friedman Omnibus Test ===")
print(f"p-value: {res['p']:.4e} | Kendall's W: {res['w']:.4f}")

if res['p'] < 0.05:
    print("\n=== Top Pairwise Comparisons (Sorted by Absolute Effect Size) ===")
    posthoc = pairwise_wilcoxon_holm(recall_matrix)
    # Sorting by absolute effect size to show largest differences
    posthoc['abs_r'] = posthoc['Rank-biserial r'].abs()
    posthoc_sorted = posthoc.sort_values('abs_r', ascending=False).drop(columns=['abs_r'])
    display(posthoc_sorted)
else:
    print("\nNo significant difference detected.")

=== Mean Recall with 95% Confidence Intervals ===


,Model,mean,std,CI_margin,CI_Lower,CI_Upper
7,Logistic Regression,0.831064,0.013373,0.016605,0.814459,0.847669
8,MLP,0.821267,0.038674,0.048020,0.773247,0.869287
1,CNN,0.819982,0.026239,0.032580,0.787402,0.852562
10,SVM,0.817269,0.049806,0.061842,0.755427,0.879112
0,Ada Boost,0.792797,0.037191,0.046179,0.746618,0.838976
11,Stacking,0.783453,0.040299,0.050038,0.733415,0.833491
2,CatBoost,0.772857,0.031840,0.039535,0.733322,0.812392
9,Random Forest,0.772454,0.045189,0.056110,0.716344,0.828563
4,Gradient Boosting,0.769792,0.024402,0.030299,0.739493,0.800091
6,LightGBM,0.769615,0.034418,0.042736,0.726879,0.812351



=== Friedman Omnibus Test ===
p-value: 2.2427e-05 | Kendall's W: 0.6895

=== Top Pairwise Comparisons (Sorted by Absolute Effect Size) ===


,Model 1,Model 2,Rank-biserial r,p_raw,p_adj,Significant
6,Ada Boost,Logistic Regression,-1.000000,0.0625,1.0,False
23,CNN,TabNet,1.000000,0.0625,1.0,False
25,CatBoost,Decision Tree,1.000000,0.0625,1.0,False
20,CNN,Random Forest,1.000000,0.0625,1.0,False
22,CNN,Stacking,1.000000,0.0625,1.0,False
...,...,...,...,...,...,...
19,CNN,MLP,-0.066667,1.0000,1.0,False
47,Gradient Boosting,LightGBM,-0.066667,1.0000,1.0,False
28,CatBoost,LightGBM,0.066667,1.0000,1.0,False
67,LightGBM,Stacking,-0.066667,1.0000,1.0,False


In [ ]:
kfold_results_df.groupby('Model')[['Accuracy', 'Precision', 'Recall', 'F1 Score']].agg(['mean', 'std'])

Accuracy           Precision              Recall  \
                         mean       std      mean       std      mean   
Model                                                                   
Ada Boost            0.887822  0.019226  0.738467  0.033935  0.792797   
CNN                  0.869325  0.012462  0.715935  0.019349  0.814592   
CatBoost             0.914083  0.016254  0.801099  0.045367  0.772857   
Decision Tree        0.853831  0.016054  0.677229  0.022051  0.725436   
Gradient Boosting    0.900952  0.014550  0.764620  0.032526  0.769792   
KNN                  0.747027  0.022709  0.618902  0.010907  0.741196   
LightGBM             0.912292  0.007477  0.794923  0.017738  0.769615   
Logistic Regression  0.874701  0.012489  0.725405  0.016164  0.831064   
MLP                  0.862170  0.013277  0.709482  0.019515  0.819506   
Random Forest        0.909900  0.012413  0.786041  0.029811  0.772454   
SVM                  0.893200  0.013228  0.748438  0.027050  0.817269   
Stacking             0.909902  0.014852  0.785821  0.036575  0.783453   
TabNet               0.788786  0.047384  0.626297  0.028558  0.708106   
XGBoost              0.907516  0.018046  0.788079  0.051110  0.762513   

                               F1 Score            
                          std      mean       std  
Model                                              
Ada Boost            0.037191  0.760876  0.035679  
CNN                  0.048357  0.747722  0.024668  
CatBoost             0.031840  0.785668  0.037173  
Decision Tree        0.035770  0.694582  0.024489  
Gradient Boosting    0.024402  0.766783  0.027751  
KNN                  0.021253  0.623580  0.017070  
LightGBM             0.034418  0.780499  0.023741  
Logistic Regression  0.013373  0.760544  0.017194  
MLP                  0.040113  0.742700  0.021604  
Random Forest        0.045189  0.778204  0.037496  
SVM                  0.049806  0.775417  0.034551  
Stacking             0.040299  0.783847  0.036183  
TabNet               0.050771  0.637149  0.031648  
XGBoost              0.030627  0.772098  0.033033

In [ ]:
import pandas as pd
import numpy as np

# Get all unique model names from the posthoc DataFrame
models = pd.concat([posthoc['Model 1'], posthoc['Model 2']]).unique()

# Create a square DataFrame filled with NaN
p_value_matrix = pd.DataFrame(index=models, columns=models, dtype=float)

# Populate the matrix with p-values
for index, row in posthoc.iterrows():
    model1 = row['Model 1']
    model2 = row['Model 2']
    p_value = row['p_raw'] # Using p_raw for the matrix as requested, if p_adj is preferred, change this line

    # Fill both symmetric entries in the matrix
    p_value_matrix.loc[model1, model2] = p_value
    p_value_matrix.loc[model2, model1] = p_value # Wilcoxon test is symmetric

# Define a highlighting function
def highlight_significant(val, threshold=0.0625):
    # Check if val is a number before comparison
    if pd.isna(val):
        return ''
    color = '#ffe6e6' if val <= threshold else ''
    return 'background-color: %s' % color

# Display the p-value matrix, highlighting significant p-values
print("2D Table of Wilcoxon Signed-Rank Test P-values (Recall):")
display(p_value_matrix.style.format("{:.4f}").map(highlight_significant))

2D Table of Wilcoxon Signed-Rank Test P-values (Recall):


,Ada Boost,CNN,CatBoost,Decision Tree,Gradient Boosting,KNN,LightGBM,Logistic Regression,MLP,Random Forest,SVM,Stacking,TabNet,XGBoost
Ada Boost,nan,0.3125,0.3125,0.1250,0.1250,0.1250,0.3125,0.0625,0.1875,0.3125,0.4375,0.8125,0.0625,0.0625
CNN,0.3125,nan,0.1250,0.0625,0.1250,0.1250,0.1250,0.4375,1.0000,0.0625,1.0000,0.0625,0.0625,0.0625
CatBoost,0.3125,0.1250,nan,0.0625,0.8125,0.1250,1.0000,0.0625,0.0625,0.6250,0.1250,0.1875,0.0625,0.3125
Decision Tree,0.1250,0.0625,0.0625,nan,0.1250,0.8125,0.1875,0.0625,0.0625,0.0625,0.0625,0.0625,0.3125,0.1250
Gradient Boosting,0.1250,0.1250,0.8125,0.1250,nan,0.1250,1.0000,0.0625,0.0625,0.6250,0.1250,0.3125,0.0625,0.1875
KNN,0.1250,0.1250,0.1250,0.8125,0.1250,nan,0.4375,0.0625,0.0625,0.1875,0.1250,0.1250,0.1875,0.3125
LightGBM,0.3125,0.1250,1.0000,0.1875,1.0000,0.4375,nan,0.0625,0.1250,1.0000,0.0625,1.0000,0.1250,0.6250
Logistic Regression,0.0625,0.4375,0.0625,0.0625,0.0625,0.0625,0.0625,nan,0.6250,0.0625,0.6250,0.0625,0.0625,0.0625
MLP,0.1875,1.0000,0.0625,0.0625,0.0625,0.0625,0.1250,0.6250,nan,0.0625,0.8125,0.0625,0.0625,0.0625
Random Forest,0.3125,0.0625,0.6250,0.0625,0.6250,0.1875,1.0000,0.0625,0.0625,nan,0.0625,0.4375,0.0625,0.4375


In [ ]:
kfold_results_df[(kfold_results_df['Model']=='Logistic Regression') & (kfold_results_df['Fold']==1)]['Recall']

,Recall
0,0.831757


In [4]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
display(posthoc_sorted)
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')

,Model 1,Model 2,Rank-biserial r,p_raw,p_adj,Significant
6,Ada Boost,Logistic Regression,-1.000000,0.0625,1.0,False
23,CNN,TabNet,1.000000,0.0625,1.0,False
25,CatBoost,Decision Tree,1.000000,0.0625,1.0,False
20,CNN,Random Forest,1.000000,0.0625,1.0,False
22,CNN,Stacking,1.000000,0.0625,1.0,False
16,CNN,KNN,1.000000,0.0625,1.0,False
15,CNN,Gradient Boosting,1.000000,0.0625,1.0,False
14,CNN,Decision Tree,1.000000,0.0625,1.0,False
13,CNN,CatBoost,1.000000,0.0625,1.0,False
12,Ada Boost,XGBoost,1.000000,0.0625,1.0,False
